# Experiment 2 — Per-patient evaluation + bootstrap 95% CI

Notebook này **không train lại model**. Nó đọc OOF predictions được tạo bởi **5-fold StratifiedGroupKFold theo patient_id** từ notebook Experiment 1.

Mục tiêu:
1. tính metric voxel-level **riêng cho từng patient_id**;
2. tổng hợp mean ± SD, median và IQR;
3. bootstrap theo **patient_id**, không bootstrap từng voxel, để tạo 95% CI cho trung bình metric theo bệnh nhân.


In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_BOOT = 5000
THRESHOLD = 0.50

# OOF chính thức do Experiment 1 xuất ra.
OOF_PATH = "output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv"
OUTPUT_DIR = "output/577p/glcm/kernel5/per_patient_bootstrap_ci"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(OOF_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy OOF: {OOF_PATH}\n"
        "Hãy chạy notebook 01_XGBoost_GLCM_ClassWeight_Ablation_FIXED.ipynb trước."
    )

print("OOF source:", OOF_PATH)


OOF source: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv


## 1. Load OOF predictions

In [2]:
oof = pd.read_csv(OOF_PATH)

PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in oof.columns), None)

required = {"label", "prob_tumor"}
if patient_col is None:
    raise ValueError("OOF file thiếu patient_id.")
if not required.issubset(oof.columns):
    raise ValueError(f"OOF file thiếu cột: {required - set(oof.columns)}")

oof[patient_col] = oof[patient_col].astype(str)
oof["label"] = oof["label"].astype(int)

# Xác nhận OOF đến từ StratifiedGroupKFold nếu metadata có sẵn.
if "cv_splitter" in oof.columns:
    splitters = set(oof["cv_splitter"].dropna().astype(str).unique())
    if splitters != {"StratifiedGroupKFold"}:
        raise RuntimeError(
            f"OOF cv_splitter={splitters}; cần chạy lại notebook 01 bằng StratifiedGroupKFold."
        )
else:
    print("WARNING: OOF chưa có cột cv_splitter; không thể xác minh splitter từ file.")

print("OOF rows:", len(oof))
print("Patients:", oof[patient_col].nunique())
print(oof["label"].value_counts().sort_index())

OOF rows: 653397
Patients: 577
0    611803
1     41594
Name: label, dtype: int64


## 2. Per-patient metrics

In [3]:
def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [4]:
patient_rows = []

for pid, g in oof.groupby(patient_col, sort=True):
    y = g["label"].to_numpy(dtype=int)
    p = g["prob_tumor"].to_numpy(dtype=float)
    m = binary_metrics(y, p, threshold=THRESHOLD)

    patient_rows.append({
        patient_col: pid,
        "n_voxels": len(g),
        "n_non_tumor": int((y == 0).sum()),
        "n_tumor": int((y == 1).sum()),
        "n_classes": int(np.unique(y).size),
        **m
    })

patient_metrics = pd.DataFrame(patient_rows)
patient_metrics.to_csv(os.path.join(OUTPUT_DIR, "per_patient_metrics.csv"), index=False)

print("Patients with both classes:", int((patient_metrics["n_classes"] == 2).sum()), "/", len(patient_metrics))
display(patient_metrics.head())

Patients with both classes: 577 / 577


,patient_id,n_voxels,n_non_tumor,n_tumor,n_classes,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,BraTS2021_00000,1112,1069,43,2,0.982014,0.789762,0.925926,0.581395,0.998129,0.714286,0.725936,0.981574,0.845604,1067,2,18,25
1,BraTS2021_00002,1054,913,141,2,0.933586,0.853720,0.755396,0.744681,0.962760,0.750000,0.711733,0.972781,0.859859,879,34,36,105
2,BraTS2021_00003,1214,1142,72,2,0.899506,0.940078,0.369792,0.986111,0.894046,0.537879,0.569749,0.994941,0.959458,1021,121,1,71
3,BraTS2021_00005,961,866,95,2,0.961498,0.959894,0.733871,0.957895,0.961894,0.831050,0.818908,0.993497,0.950390,833,33,4,91
4,BraTS2021_00006,1131,1027,104,2,0.925729,0.954783,0.553763,0.990385,0.919182,0.710345,0.709029,0.997247,0.985047,944,83,1,103


## 3. Descriptive summary

In [5]:
metric_cols = [
    "accuracy","balanced_accuracy","precision","recall","specificity",
    "f1","mcc","roc_auc","pr_auc"
]

summary_rows = []
for metric in metric_cols:
    x = patient_metrics[metric].dropna().astype(float)
    summary_rows.append({
        "metric": metric,
        "n_patients_valid": len(x),
        "mean": x.mean(),
        "sd": x.std(ddof=1),
        "median": x.median(),
        "q1": x.quantile(0.25),
        "q3": x.quantile(0.75)
    })

descriptive = pd.DataFrame(summary_rows)
descriptive.to_csv(os.path.join(OUTPUT_DIR, "per_patient_descriptive_summary.csv"), index=False)
display(descriptive)

,metric,n_patients_valid,mean,sd,median,q1,q3
0,accuracy,577,0.916163,0.114785,0.958521,0.899535,0.979695
1,balanced_accuracy,577,0.903498,0.082631,0.931894,0.869055,0.964600
2,precision,577,0.574403,0.244979,0.634783,0.394737,0.761905
3,recall,577,0.890971,0.148444,0.948454,0.863636,0.985075
4,specificity,577,0.916026,0.126376,0.966002,0.899602,0.985075
5,f1,577,0.652735,0.209911,0.710345,0.540943,0.816667
6,mcc,577,0.658976,0.193793,0.702470,0.556040,0.809478
7,roc_auc,577,0.981081,0.027178,0.991371,0.977430,0.996430
8,pr_auc,577,0.858766,0.138273,0.902762,0.821231,0.948136


## 4. Bootstrap 95% CI theo patient_id

Mỗi bootstrap replicate lấy mẫu **bệnh nhân với hoàn lại** rồi tính **trung bình metric theo bệnh nhân**.
Vì vậy bệnh nhân, không phải voxel, là đơn vị bootstrap.

In [6]:
rng = np.random.default_rng(RANDOM_STATE)
n_patients = len(patient_metrics)

boot_rows = []
for b in range(N_BOOT):
    idx = rng.integers(0, n_patients, size=n_patients)
    sample = patient_metrics.iloc[idx]

    row = {"bootstrap": b + 1}
    for metric in metric_cols:
        row[metric] = sample[metric].dropna().mean()
    boot_rows.append(row)

boot = pd.DataFrame(boot_rows)
boot.to_csv(os.path.join(OUTPUT_DIR, "patient_bootstrap_replicates.csv"), index=False)

ci_rows = []
for metric in metric_cols:
    x = boot[metric].dropna().to_numpy(dtype=float)
    ci_rows.append({
        "metric": metric,
        "bootstrap_n": len(x),
        "point_estimate_mean_patient_metric": patient_metrics[metric].dropna().mean(),
        "ci95_low_percentile": np.quantile(x, 0.025),
        "ci95_high_percentile": np.quantile(x, 0.975)
    })

ci = pd.DataFrame(ci_rows)
ci.to_csv(os.path.join(OUTPUT_DIR, "per_patient_bootstrap_95ci.csv"), index=False)
display(ci)

,metric,bootstrap_n,point_estimate_mean_patient_metric,ci95_low_percentile,ci95_high_percentile
0,accuracy,5000,0.916163,0.906467,0.925522
1,balanced_accuracy,5000,0.903498,0.896759,0.910269
2,precision,5000,0.574403,0.554005,0.594013
3,recall,5000,0.890971,0.878790,0.902570
4,specificity,5000,0.916026,0.905390,0.926224
5,f1,5000,0.652735,0.635270,0.669946
6,mcc,5000,0.658976,0.642871,0.674655
7,roc_auc,5000,0.981081,0.978748,0.983245
8,pr_auc,5000,0.858766,0.847194,0.869884


## Reporting note

Các CI này là **CI của trung bình metric voxel-level tính theo từng bệnh nhân**.
Không gọi chúng là patient-level disease classification metrics.